Run IMSTE on MNIST, Fashion-MNIST, and Kuzushiji-MNIST. For each dataset, the gallery first shows a grid of sample images with one column per class and a configurable number of rows. It standardizes a stratified sample and fits IMSTE directly to the flattened image features. Multilevel initialization is enabled by default in this notebook; it builds a rank-only coarse-to-fine starting layout before the final IMSTE fit. Choose the MST builder with `MST_METHOD`: FAMST is the approximate default, and Prim builds the exact graph. A following cell fits UMAP to those same standardized samples and compares IMSTE and UMAP side by side, with each method's runtime in its title. The summary reports the sample and feature counts and selected builder.

Remote datasets download on first use and are cached. Set `MAX_SAMPLES` or `N_EPOCHS` lower for a faster run. `N_MSTS`, `NEGATIVE_RATIO`, and `FAMST_MAX_NEIGHBORS` configure the graph and negative sampling; their defaults are 4, 1, and 15. Change `MULTILEVEL_INIT` to `False` to compare the existing PCA initialization, or adjust the multilevel reduction, base size, and per-level epoch budget below.

In [51]:
%load_ext autoreload
%autoreload 2

%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import sys

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "notebooks" / "src" / "high_dim_mst_gallery.py").is_file()),
    None,
)
if repo_root is not None:
    sys.path.insert(0, str(repo_root / "notebooks" / "src"))
    sys.path.insert(0, str(repo_root))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from high_dim_mst_gallery import run_high_dim_mst_gallery

# Dataset
MAX_SAMPLES = 60000
SAMPLE_PLOT_ROWS = 5

#----
# Graph construction
MST_METHOD = "famst"  # choose "prim" for the exact graph famst
N_MSTS = 16
NEGATIVE_RATIO = 1
FAMST_MAX_NEIGHBORS = 30
MULTILEVEL_INIT = True
MULTILEVEL_REDUCTION_RATIO = 4
MULTILEVEL_BASE_SIZE = 1024
MULTILEVEL_EPOCHS = 10

# Class-mixture graph
MAX_MIXTURE_COMPONENTS_PER_CLASS = 4
MIN_MIXTURE_COMPONENT_SIZE = 20

gallery = run_high_dim_mst_gallery(
    max_samples=MAX_SAMPLES,
    sample_plot_rows=SAMPLE_PLOT_ROWS,
    n_msts=N_MSTS,
    negative_ratio=NEGATIVE_RATIO,
    mst_max_neighbors=FAMST_MAX_NEIGHBORS,
    mst_method=MST_METHOD,
    multilevel_init=MULTILEVEL_INIT,
    multilevel_reduction_ratio=MULTILEVEL_REDUCTION_RATIO,
    multilevel_base_size=MULTILEVEL_BASE_SIZE,
    multilevel_epochs=MULTILEVEL_EPOCHS,
    n_epochs=1000,
    plot_embeddings=False,  # Compare IMSTE and UMAP together in the next cell.
)
gallery["summary"]


## Side-by-side IMSTE and UMAP embeddings

Fit UMAP to the same standardized samples used for IMSTE, then compare each dataset in a shared figure. Both panels use the same class labels and report their fit times in the titles.


In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import umap
from IPython.display import display
from sklearn.preprocessing import StandardScaler

UMAP_RESULTS = {}
IMSTE_SECONDS = gallery["summary"].set_index("dataset")["seconds"].to_dict()
for dataset_name, (raw_X, labels) in gallery["datasets"].items():
    X_standardized = StandardScaler().fit_transform(raw_X)
    started = perf_counter()
    umap_embedding = umap.UMAP(
        n_components=2, n_neighbors=15, min_dist=0.1,
        metric="euclidean", random_state=42, n_jobs=1,
    ).fit_transform(X_standardized)
    umap_seconds = perf_counter() - started
    UMAP_RESULTS[dataset_name] = umap_embedding

    imste_embedding = gallery["embeddings"][dataset_name]
    fig, (ax_imste, ax_umap, colorbar_axis) = plt.subplots(
        1, 3,
        figsize=(14, 5),
        gridspec_kw={"width_ratios": [1, 1, 0.04]},
        constrained_layout=True,
    )
    points = None
    for ax, embedding, method, elapsed in (
        (ax_imste, imste_embedding, "IMSTE", IMSTE_SECONDS[dataset_name]),
        (ax_umap, umap_embedding, "UMAP", umap_seconds),
    ):
        points = ax.scatter(
            embedding[:, 0], embedding[:, 1],
            c=labels, cmap="tab20", vmin=0, vmax=9,
            s=12, alpha=0.8, linewidths=0,
        )
        ax.set_title(f"{method} — fit: {elapsed:.2f} s")
        ax.set_xlabel(f"{method} dimension 1")
        ax.set_ylabel(f"{method} dimension 2")
    fig.suptitle(dataset_name)
    fig.colorbar(
        points,
        cax=colorbar_axis,
        label="Class label",
        ticks=np.unique(labels),
    )
    display(fig)
    plt.close(fig)


## Class-mixture graph overlay

The reusable plotting function in `imste.class_graph` fits up to `MAX_MIXTURE_COMPONENTS_PER_CLASS` full-covariance Gaussian components per class, with at least `MIN_MIXTURE_COMPONENT_SIZE` samples per component in each 2D embedding. Each component mean becomes a node; node size reflects mixture weight and node color identifies its class. Two exact iterated MSTs are built over the nodes, then same-class edges are omitted. Remaining edges are black and match the node-outline width. Their opacity increases with symmetric cross-component probability: the two endpoint Gaussian densities form an equal-prior pairwise posterior, evaluated on the other component’s assigned samples in both directions. The directional probabilities are averaged; chance-level overlap (0.5) is fully opaque. All graph scores use the 2D embedding.

In [ ]:
import pandas as pd
from imste.class_graph import plot_class_mixture_mst

class_graph_edge_tables = []
for dataset_name, (raw_X, labels) in gallery["datasets"].items():
    fig, _, edge_scores = plot_class_mixture_mst(
        raw_X,
        labels,
        {
            "IMSTE": gallery["embeddings"][dataset_name],
            "UMAP": UMAP_RESULTS[dataset_name],
        },
        title=dataset_name,
        n_msts=2,
        max_components_per_class=4,
        min_component_size=80,
        point_alpha=0.4,
        min_edge_alpha=0.2,
        max_edge_alpha=0.9,
        random_state=42,
    )
    display(fig)
    plt.close(fig)
    edge_scores.insert(0, "dataset", dataset_name)
    class_graph_edge_tables.append(edge_scores)

class_graph_edges = pd.concat(class_graph_edge_tables, ignore_index=True)
class_graph_edges.groupby(["dataset", "embedding"]).agg(
    edges=("edge_alpha", "size"),
    mean_edge_alpha=("edge_alpha", "mean"),
)

## MNIST relations unique to each embedding

An edge connects two class-specific Gaussian components, so we collapse component edges to unordered digit pairs. The table lists pairs selected by the IMSTE MST but not the UMAP MST, and vice versa. It ranks each direction by mean edge error across distinct component edges: higher values mean the two endpoint Gaussians are harder to distinguish in that embedding. Component details and the maximum edge error are retained for inspection.

Use these as candidate relationships to interpret, not automatic evidence that a pair is semantically related. For example, a strong 4–9 relation is plausible because those digits can be visually confusable; check surprising pairs against digit examples or an independent MNIST confusion matrix. A missing edge only means that pair was not selected by that embedding’s MST.

In [ ]:
from IPython.display import display
from imste.class_graph import compare_class_graph_relations

mnist_relation_differences = compare_class_graph_relations(
    class_graph_edges,
    method_a="IMSTE",
    method_b="UMAP",
    dataset="MNIST (28×28)",
)
display(mnist_relation_differences)

## Datashader rasterized embeddings

The scatter plots above are useful for this 10,000-row sample. For larger samples, the reusable `render_datashader_embedding` helper aggregates points per pixel to avoid overplotting. This display uses a higher `min_alpha` and fixed `spread_px=2` to make each mark more visible; set `spread_px=0` to disable spreading. Try `spread_px=1` or `3` to tune the mark size. For density-aware spreading instead, use `max_spread_px` (and leave `spread_px=0`). Lower `min_alpha` for very dense datasets. Set `DATASHADER_BACKGROUND` to `"white"` or `"black"` to compare contrast. The helper also supports density-only views and zoom ranges through `x_range` / `y_range`.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from imste.visualization import render_datashader_embedding

DATASHADER_BACKGROUND = "white"  # Change to "white" to compare.
DATASHADER_IMAGES = {}
for dataset_name, (_, labels) in gallery["datasets"].items():
    embeddings = {
        "IMSTE": gallery["embeddings"][dataset_name],
        "UMAP": UMAP_RESULTS[dataset_name],
    }
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
    for ax, (method, embedding) in zip(axes, embeddings.items()):
        image = render_datashader_embedding(
            embedding,
            labels,
            width=700,
            height=700,
            min_alpha=40,
            background=DATASHADER_BACKGROUND,
            spread_px=1,  # Fixed mark radius in pixels; try 1 or 3.

        )
        DATASHADER_IMAGES[(dataset_name, method)] = image
        ax.imshow(image.to_pil())
        ax.set_title(f"{dataset_name} — {method}")
        ax.set_axis_off()
    display(fig)
    plt.close(fig)

# Zoom example: choose bounds in the coordinate system of one embedding.
# zoomed = render_datashader_embedding(
#     gallery["embeddings"]["MNIST (28×28)"],
#     gallery["datasets"]["MNIST (28×28)"][1],
#     x_range=(-2, 2), y_range=(-2, 2), width=900, height=700,
# )
# display(zoomed)